# MVTec Anomaly Detection — All Categories
**Pipeline:** Preprocessing → Harris → Pyramid → SIFT → Segmentation → Classification

اختاري الكاتيجوري من الـ Gradio وهو هيعمل كل حاجة أوتوماتيك.

**Dataset:** `/kaggle/input/mvtec-ad/`

## 0. Setup & Imports

In [1]:
import os
import glob
import numpy as np

# ── Matplotlib backend لازم يتعمل قبل أي import تاني ─────────────────────
import matplotlib
matplotlib.use('Agg')          # بدون GUI — مهم جداً في Kaggle + Gradio
import matplotlib.pyplot as plt

import cv2
import io
from PIL import Image
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix
)
from sklearn.preprocessing import StandardScaler
import copy
import warnings
warnings.filterwarnings('ignore')

# ── Paths ─────────────────────────────────────────────────────────────────
# عدّلي المسار لو الداتاست في مكان تاني
DATASET_ROOT = '/kaggle/input/datasets/ipythonx/mvtec-ad'

ALL_CATEGORIES = [
    'bottle', 'cable', 'capsule', 'carpet', 'grid',
    'hazelnut', 'leather', 'metal_nut', 'pill',
    'screw', 'tile', 'toothbrush', 'transistor',
    'wood', 'zipper'
]

IMG_SIZE = (256, 256)
SEED     = 42
np.random.seed(SEED)

sift = cv2.SIFT_create(nfeatures=300)

print('✓ Imports done')
print(f'Dataset root : {DATASET_ROOT}')
print(f'Exists       : {os.path.exists(DATASET_ROOT)}')

✓ Imports done
Dataset root : /kaggle/input/datasets/ipythonx/mvtec-ad
Exists       : True


## 1. Core Pipeline Functions

In [2]:
# ─────────────────────────────────────────────────────────────────────────
# DATA LOADING
# ─────────────────────────────────────────────────────────────────────────
def load_images(folder, label):
    paths = sorted(glob.glob(os.path.join(folder, '*.png')))
    imgs, labels = [], []
    for p in paths:
        img = cv2.imread(p)
        if img is None:
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, IMG_SIZE)
        imgs.append(img)
        labels.append(label)
    return imgs, labels


def load_category(category):
    base       = os.path.join(DATASET_ROOT, category)
    train_good = os.path.join(base, 'train', 'good')
    test_dir   = os.path.join(base, 'test')

    if not os.path.isdir(train_good):
        raise FileNotFoundError(f'مسار التدريب مش موجود: {train_good}')

    train_imgs, train_labels = load_images(train_good, label=0)

    test_imgs, test_labels, defect_types = [], [], []
    if os.path.isdir(test_dir):
        defect_types = sorted([
            d for d in os.listdir(test_dir)
            if os.path.isdir(os.path.join(test_dir, d))
        ])
        for dt in defect_types:
            lbl  = 0 if dt == 'good' else 1
            imgs, lbls = load_images(os.path.join(test_dir, dt), label=lbl)
            test_imgs   += imgs
            test_labels += lbls

    return train_imgs, train_labels, test_imgs, test_labels, defect_types


# ─────────────────────────────────────────────────────────────────────────
# PREPROCESSING HELPERS
# ─────────────────────────────────────────────────────────────────────────
def harris_corners(img_rgb, threshold=0.01):
    gray     = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY).astype(np.float32)
    response = cv2.cornerHarris(gray, blockSize=2, ksize=3, k=0.04)
    response = cv2.dilate(response, None)
    marked   = img_rgb.copy()
    marked[response > threshold * response.max()] = [255, 0, 0]
    count    = int(np.sum(response > threshold * response.max()))
    return marked, count


def kmeans_segment(img_rgb, k=3):
    # ملحوظة: K-Means بتشتغل على الصورة الملونة RGB
    # مش الـ grayscale — عشان كده بنقدر نستخرج color-based clusters
    pixels   = img_rgb.reshape(-1, 3).astype(np.float32)
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 100, 0.2)
    _, labels, centers = cv2.kmeans(
        pixels, k, None, criteria, 10, cv2.KMEANS_RANDOM_CENTERS
    )
    centers  = centers.astype(np.uint8)
    seg_img  = centers[labels.flatten()].reshape(img_rgb.shape)
    lbl_mask = labels.reshape(img_rgb.shape[:2])
    return seg_img, lbl_mask


# ─────────────────────────────────────────────────────────────────────────
# FEATURE EXTRACTION — 22 features per image
#
# ملحوظة عن الـ SIFT:
#   SIFT بيطلع descriptor (128,) لكل keypoint.
#   لو عندنا 200 keypoint → matrix (200, 128).
#   مش ممكن نحط الـ 25600 رقم ده كـ features مباشرة لأن:
#     1) الحجم متغير (كل صورة عندها عدد keypoints مختلف)
#     2) هيبقى أكبر بكتير من الداتا اللي عندنا
#   الحل: بناخد mean + std على كل الـ matrix → رقمين بس
#   ده اسمه "global descriptor statistics" وشائع جداً في CV التقليدي.
#
# ملحوظة عن الـ Color Features:
#   Harris، SIFT، Laplacian → بيشتغلوا على grayscale
#   لكن Color Stats (r_mean, g_mean, ...) → بتاخد من img_rgb الأصلية
#   الصورة الملونة موجودة، بس بنحولها grayscale بس لما نحتاجها
# ─────────────────────────────────────────────────────────────────────────
def extract_features(img_rgb):
    # grayscale للعمليات اللي محتاجاها
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY).astype(np.float32)

    # 1. Filter difference stats (4 features)
    gauss  = cv2.GaussianBlur(gray, (5, 5), 1.5)
    median = cv2.medianBlur(gray.astype(np.uint8), 5).astype(np.float32)
    diff_g = np.abs(gray - gauss)
    diff_m = np.abs(gray - median)

    # 2. Harris corner density (1 feature)
    response       = cv2.cornerHarris(gray, 2, 3, 0.04)
    corner_density = float(np.sum(response > 0.01 * response.max())) / (IMG_SIZE[0] * IMG_SIZE[1])

    # 3. SIFT stats (5 features)
    #    - n_kps           : عدد الـ keypoints
    #    - resp_mean/std   : إحصائيات قوة الـ keypoints
    #    - desc_mean/std   : إحصائيات على مصفوفة الـ descriptors (N×128)
    kps, descs = sift.detectAndCompute(gray.astype(np.uint8), None)
    n_kps      = len(kps)
    resp_mean  = float(np.mean([k.response for k in kps])) if kps else 0.0
    resp_std   = float(np.std ([k.response for k in kps])) if kps else 0.0
    desc_mean  = float(descs.mean()) if descs is not None else 0.0
    desc_std   = float(descs.std())  if descs is not None else 0.0

    # 4. Laplacian edge energy (3 features)
    lap      = cv2.Laplacian(gray.astype(np.uint8), cv2.CV_64F)
    lap_mean = float(np.mean(np.abs(lap)))
    lap_std  = float(np.std(lap))
    lap_var  = float(lap.var())

    # 5. Color stats من الصورة الملونة الأصلية (6 features)
    #    Harris/SIFT/LAP بيشتغلوا على grayscale
    #    لكن img_rgb لسه موجودة وبنستخرج منها الـ color info
    r_mean, g_mean, b_mean = [float(img_rgb[:, :, c].mean()) for c in range(3)]
    r_std,  g_std,  b_std  = [float(img_rgb[:, :, c].std())  for c in range(3)]

    # 6. K-Means cluster ratios من الصورة الملونة (3 features)
    #    K-Means على RGB → بيقسم الصورة حسب الألوان
    _, lbl     = kmeans_segment(img_rgb, k=4)
    counts     = [np.sum(lbl == i) for i in range(3)]
    seg_ratios = [c / lbl.size for c in sorted(counts)]

    # Total: 4 + 1 + 5 + 3 + 6 + 3 = 22 features
    return np.array([
        diff_g.mean(), diff_g.std(), diff_m.mean(), diff_m.std(),   # 4
        corner_density,                                              # 1
        n_kps, resp_mean, resp_std, desc_mean, desc_std,            # 5
        lap_mean, lap_std, lap_var,                                  # 3
        r_mean, g_mean, b_mean, r_std, g_std, b_std,                # 6
        *seg_ratios                                                  # 3
    ], dtype=np.float32)


# ─────────────────────────────────────────────────────────────────────────
# TRAIN PIPELINE
# ─────────────────────────────────────────────────────────────────────────
def train_pipeline(category):
    print(f'=== Loading category: {category} ===')
    train_imgs, train_labels, test_imgs, test_labels, defect_types = load_category(category)
    print(f'  Train: {len(train_imgs)}  |  Test: {len(test_imgs)}')

    if len(train_imgs) == 0:
        raise ValueError(f'مفيش صور في train/good لـ {category}')

    all_imgs   = train_imgs + test_imgs
    all_labels = train_labels + test_labels

    print('  Extracting features...')
    X = np.array([extract_features(img) for img in all_imgs])
    y = np.array(all_labels)
    print(f'  Feature matrix: {X.shape}')

    # تأكد إن عندنا كلتين الـ classes قبل stratify
    stratify_y = y if len(np.unique(y)) > 1 else None

    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.25, random_state=SEED, stratify=stratify_y
    )

    # ── Bug Fix 1: scaler بيتعمل fit على X_train بس ─────────────────────
    scaler   = StandardScaler()
    X_tr_s   = scaler.fit_transform(X_tr)
    X_te_s   = scaler.transform(X_te)

    classifiers = {
        'Naive Bayes'      : GaussianNB(),
        'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=SEED),
        'Random Forest'    : RandomForestClassifier(n_estimators=100, random_state=SEED),
    }

    results = {}
    trained_clfs = {}
    for name, clf in classifiers.items():
        clf.fit(X_tr_s, y_tr)
        y_pred = clf.predict(X_te_s)
        results[name] = {
            'acc' : accuracy_score (y_te, y_pred),
            'prec': precision_score(y_te, y_pred, zero_division=0),
            'rec' : recall_score   (y_te, y_pred, zero_division=0),
            'f1'  : f1_score       (y_te, y_pred, zero_division=0),
            'cm'  : confusion_matrix(y_te, y_pred),
        }
        trained_clfs[name] = clf   # ── Bug Fix 2: نفصل الـ clf عن results ──
        print(f'  [{name}]  Acc={results[name]["acc"]:.3f}  F1={results[name]["f1"]:.3f}')

    best_name = max(results, key=lambda k: results[k]['f1'])
    print(f'  Best: {best_name}  (F1={results[best_name]["f1"]:.3f})')

    # ── Bug Fix 3: إعادة التدريب على كل الداتا بشكل صح ──────────────────
    # بناخد نسخة جديدة من الموديل بدل ما نعيد استخدام الـ fitted واحد
    # والـ scaler بيتعمل fit على كل X مرة واحدة بس هنا
    best_clf = copy.deepcopy(trained_clfs[best_name].__class__(
        **trained_clfs[best_name].get_params()
    ))
    X_all_s = scaler.fit_transform(X)   # الـ scaler بيتحدث على كل الداتا
    best_clf.fit(X_all_s, y)
    print('  Model re-trained on full dataset ✓')

    return best_clf, scaler, results, best_name, defect_types


print('✓ Core functions loaded')

✓ Core functions loaded


## 2. Gradio Interface

In [3]:
!pip install gradio -q

In [4]:
import gradio as gr

# ── Global state ──────────────────────────────────────────────────────────
_state = {
    'category'  : None,
    'clf'       : None,
    'scaler'    : None,
    'results'   : None,
    'best_name' : None,
    'defects'   : [],
}


def fig_to_pil(fig):
    """Matplotlib figure → PIL Image (للـ Gradio)."""
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=130, bbox_inches='tight')
    buf.seek(0)
    img = Image.open(buf).copy()
    plt.close(fig)
    return img


# ─────────────────────────────────────────────────────────────────────────
# STEP 1 — تدريب على الكاتيجوري المختارة
# ─────────────────────────────────────────────────────────────────────────
def train_on_category(category):
    """ترجع (status_text, metrics_img, cm_img)."""
    if not category:
        return 'اختاري كاتيجوري الأول!', None, None

    try:
        clf, scaler, results, best_name, defects = train_pipeline(category)
    except Exception as e:
        return f'Error أثناء التدريب:\n{e}', None, None

    _state.update({
        'category'  : category,
        'clf'       : clf,
        'scaler'    : scaler,
        'results'   : results,
        'best_name' : best_name,
        'defects'   : defects,
    })

    # ── Metrics bar chart ──────────────────────────────────────────────────
    fig1, axes = plt.subplots(1, 3, figsize=(14, 4))
    fig1.suptitle(f'Classifier Performance — {category}', fontsize=13, fontweight='bold')
    colors  = ['#4C72B0', '#DD8452', '#55A868']
    metrics = ['acc', 'prec', 'rec', 'f1']
    mlabels = ['Accuracy', 'Precision', 'Recall', 'F1']
    for ax, (cname, color) in zip(axes, zip(results.keys(), colors)):
        r    = results[cname]
        vals = [r[m] for m in metrics]
        bars = ax.bar(mlabels, vals, color=color, alpha=0.85, edgecolor='white')
        ax.set_ylim(0, 1.12)
        ax.set_title(cname, fontsize=9, fontweight='bold')
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2, val + 0.03,
                    f'{val:.2f}', ha='center', va='bottom', fontsize=9)
    plt.tight_layout()
    metrics_img = fig_to_pil(fig1)

    # ── Confusion matrices ────────────────────────────────────────────────
    fig2, axes2 = plt.subplots(1, 3, figsize=(12, 4))
    fig2.suptitle('Confusion Matrices', fontsize=13, fontweight='bold')
    for ax, (cname, color) in zip(axes2, zip(results.keys(), colors)):
        cm = results[cname]['cm']
        ax.imshow(cm, cmap='Blues')
        ax.set_title(cname, fontsize=9)
        ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
        ax.set_xticklabels(['Normal', 'Defect'])
        ax.set_yticklabels(['Normal', 'Defect'])
        ax.set_xlabel('Predicted'); ax.set_ylabel('True')
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                        color='white' if cm[i, j] > cm.max() / 2 else 'black',
                        fontsize=14)
    plt.tight_layout()
    cm_img = fig_to_pil(fig2)

    # ── Status text ───────────────────────────────────────────────────────
    r = results[best_name]
    status = (
        f'✓ تم تدريب الموديل على: {category}\n'
        f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n'
        f'أنواع العيوب : {defects}\n'
        f'\nأفضل موديل  : {best_name}\n'
        f'  Accuracy   : {r["acc"]:.4f}\n'
        f'  Precision  : {r["prec"]:.4f}\n'
        f'  Recall     : {r["rec"]:.4f}\n'
        f'  F1 Score   : {r["f1"]:.4f}\n'
        f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n'
        f'دلوقتي ارفعي صورة وحللي!'
    )
    return status, metrics_img, cm_img


# ─────────────────────────────────────────────────────────────────────────
# STEP 2 — تحليل صورة
# ─────────────────────────────────────────────────────────────────────────
def analyze_image(pil_image):
    """ترجع (result_img, report_text)."""
    if pil_image is None:
        return None, 'ارفعي صورة الأول!'
    if _state['clf'] is None:
        return None, 'دربي الموديل الأول — اختاري كاتيجوري واضغطي Train Model!'

    try:
        img_rgb = np.array(pil_image.convert('RGB'))
        img_rgb = cv2.resize(img_rgb, IMG_SIZE)

        feat   = extract_features(img_rgb).reshape(1, -1)
        feat_s = _state['scaler'].transform(feat)
        pred   = _state['clf'].predict(feat_s)[0]
        proba  = (
            _state['clf'].predict_proba(feat_s)[0]
            if hasattr(_state['clf'], 'predict_proba') else None
        )

        label_str  = 'NORMAL ✓'  if pred == 0 else 'DEFECTIVE ✗'
        label_ar   = 'سليمة'     if pred == 0 else 'معيبة'
        color_hex  = '#2d8a4e'   if pred == 0 else '#c0392b'
        confidence = f'{max(proba)*100:.1f}%' if proba is not None else 'N/A'

        # visualizations
        seg, _         = kmeans_segment(img_rgb, k=3)
        harris_m, cnt  = harris_corners(img_rgb)
        gray_u8        = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
        kps_img, _     = sift.detectAndCompute(gray_u8, None)
        sift_drawn     = cv2.drawKeypoints(
            gray_u8, kps_img, None,
            flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS
        )

        fig, axes = plt.subplots(1, 4, figsize=(18, 4))
        fig.patch.set_facecolor('#f8f9fa')
        fig.suptitle(
            f'Category: {_state["category"]}  |  {label_str} ({label_ar})'
            f'  |  Confidence: {confidence}',
            fontsize=13, fontweight='bold', color=color_hex, y=1.03
        )
        axes[0].imshow(img_rgb);                 axes[0].set_title('Input Image');                     axes[0].axis('off')
        axes[1].imshow(seg);                     axes[1].set_title('K-Means Segmentation (K=3)');      axes[1].axis('off')
        axes[2].imshow(harris_m);                axes[2].set_title(f'Harris Corners ({cnt})');         axes[2].axis('off')
        axes[3].imshow(sift_drawn, cmap='gray'); axes[3].set_title(f'SIFT Keypoints ({len(kps_img)})');axes[3].axis('off')
        plt.tight_layout()
        result_img = fig_to_pil(fig)

        proba_str = ''
        if proba is not None:
            proba_str = (
                f'  P(Normal)    = {proba[0]*100:.1f}%\n'
                f'  P(Defective) = {proba[1]*100:.1f}%\n'
            )

        report = (
            f'============================================\n'
            f'  ANOMALY DETECTION REPORT\n'
            f'============================================\n'
            f'  Category     : {_state["category"]}\n'
            f'  Prediction   : {label_str} ({label_ar})\n'
            f'  Confidence   : {confidence}\n'
            f'{proba_str}'
            f'  Harris corners : {cnt}\n'
            f'  SIFT keypoints : {len(kps_img)}\n'
            f'  Best model     : {_state["best_name"]}\n'
            f'============================================'
        )
        return result_img, report

    except Exception as e:
        return None, f'Error أثناء التحليل:\n{e}'


# ─────────────────────────────────────────────────────────────────────────
# GRADIO LAYOUT
# ─────────────────────────────────────────────────────────────────────────
with gr.Blocks(
    title='MVTec AD — Multi-Category Anomaly Detector',
    theme=gr.themes.Soft()
) as demo:

    gr.Markdown("""
    # MVTec Anomaly Detection — كل الكاتيجوريز
    **الخطوات:**
    1. اختاري الكاتيجوري (Bottle, Cable, Capsule, ...)
    2. اضغطي **Train Model** — الموديل هيتدرب أوتوماتيك
    3. ارفعي أي صورة من فولدر `test/` وهو هيحللها
    """)

    with gr.Row():
        with gr.Column(scale=2):
            cat_dropdown = gr.Dropdown(
                choices=ALL_CATEGORIES,
                label='اختاري الكاتيجوري',
                value='bottle',
            )
            train_btn = gr.Button('Train Model', variant='primary', size='lg')
        with gr.Column(scale=3):
            train_status = gr.Textbox(
                label='Training Status',
                lines=10,
                placeholder='اختاري كاتيجوري واضغطي Train...'
            )

    with gr.Row():
        metrics_out = gr.Image(label='Classifier Metrics')
        cm_out      = gr.Image(label='Confusion Matrices')

    gr.Markdown('---')
    gr.Markdown('## تحليل صورة جديدة')

    with gr.Row():
        with gr.Column(scale=1):
            inp_img     = gr.Image(type='pil', label='ارفعي صورة من test/', height=280)
            analyze_btn = gr.Button('Analyze Image', variant='secondary', size='lg')
            gr.Markdown("""
            **تيست:**
            - `test/good` → NORMAL ✓
            - باقي الفولدرات → DEFECTIVE ✗
            """)
        with gr.Column(scale=2):
            out_vis  = gr.Image(label='Visual Analysis', height=350)
            out_text = gr.Textbox(label='Detection Report', lines=12, show_copy_button=True)

    train_btn.click(
        fn=train_on_category,
        inputs=[cat_dropdown],
        outputs=[train_status, metrics_out, cm_out]
    )
    analyze_btn.click(
        fn=analyze_image,
        inputs=[inp_img],
        outputs=[out_vis, out_text]
    )

demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://dce44c01f9af90f2e9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
